# Candidate recovery metric: resting heart rate

Resting heart rate is the wearable signal least dependent on a patient choosing to be active, so
it can register recovery in someone too unwell to walk -- exactly the patients a step-count
outcome loses. It is also harder to obtain: All of Us publishes daily heart-rate zone summaries
but no resting value, so the metric has to be derived from the minute-level table.

This notebook audits coverage from the cheap daily summary first, and only then estimates what
deriving a resting value from minute-level data would cost. Cost is a real selection criterion
here, not an afterthought.

**Governance.** Every query aggregates inside BigQuery and returns counts,
quantiles or correlations only, so no participant-level rows enter the notebook. Unsuppressed
figures stay git-ignored while `*_suppressed.png` is committable, and the small-cell threshold
must be confirmed against current All of Us policy before anything is published.

# 1-setup

### Datasets, cohort SQL and output directories

`srp_common` resolves the pinned CDR and cohort index, creates this notebook's git-ignored
figure and cache directories, and builds the cohort plus one index date per patient. The
metric-specific CTE is defined here too, so the whole SQL definition this notebook runs on sits
in one place.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas

_root = Path.cwd().resolve()
while not (_root / "srp_common.py").exists():
    if _root.parent == _root:
        raise RuntimeError(
            "Repo root not found: no srp_common.py at or above "
            f"{Path.cwd().resolve()}. Run this notebook from inside the repository."
        )
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

import srp_common
from srp_common import CATEGORICAL, INK, INK_MUTED, SERIES, SUPPRESS_THRESHOLD

cfg = srp_common.init("fitbit_heart_rate")
CDR = cfg.CDR
FIGURES, CACHE_DIR = cfg.FIGURES, cfg.CACHE_DIR
cached_gbq = cfg.cached_gbq

# Which of a patient's surgery-days becomes the index date is still open --
# index_surgery/surgery_day_distribution.ipynb is what settles it. "first" is used
# until then because it is the only rule no post-operative event can influence, and
# every result below is stamped with the rule so a later change is visible.
INDEX_RULE = "first"
WINDOWS = srp_common.DEFAULT_WINDOWS

METRIC_CTE = f"""
    hr_daily AS (
        SELECT
            h.person_id,
            DATE_DIFF(h.date, i.index_date, DAY) AS rel_day,
            SUM(h.minute_in_zone)                AS minutes_recorded
        FROM `{CDR}`.heart_rate_summary AS h
        JOIN index_surgery AS i ON i.person_id = h.person_id
        GROUP BY h.person_id, rel_day
    )"""

# Deriving a resting value needs the minute-level table, which is orders of magnitude
# larger. Nothing reads it until a dry run has priced the scan and it comes in under
# this budget -- an accidental full scan is the expensive mistake in this repo.
MINUTE_LEVEL_BUDGET_GIB = 200

CTES = cfg.cohort_ctes(INDEX_RULE, extra=METRIC_CTE)

print(f"index rule : {INDEX_RULE}")
print("windows    : " + ", ".join(f"{k} [{lo}, {hi}]" for k, (lo, hi) in WINDOWS.items()))

# 2-coverage

### Do the heart-rate tables have the columns this notebook assumes?

The daily summary and the minute-level table are named and shaped differently across All of Us
releases, and a wrong column name fails late and opaquely. This probe reads schema metadata only.

In [ ]:
hr_columns = cached_gbq(
    srp_common.columns_sql(cfg, ["heart_rate_summary", "heart_rate_minute_level"]),
    "heart_rate_columns",
)

if hr_columns.empty:
    print("Neither heart-rate table was found -- check the table names in the "
          "All of Us data dictionary for this release before going further.")
else:
    for table, group in hr_columns.groupby("table_name"):
        print(f"\n{table}")
        print("  " + ", ".join(group["column_name"]))

### How many cohort patients have any heart-rate record, and on which days?

The daily summary is cheap enough to scan across the whole relative-day range, so it gives the
coverage answer without touching minute-level data. Coverage here is an upper bound on resting
heart rate, which additionally needs enough minutes in a day to estimate from.

In [ ]:
coverage_sql = CTES + """
    SELECT
        (SELECT COUNT(*) FROM cohort)                       AS cohort_patients,
        (SELECT COUNT(DISTINCT person_id) FROM hr_daily)    AS patients_with_any_hr,
        (SELECT COUNT(*) FROM hr_daily)                     AS hr_days
"""

cov = cached_gbq(coverage_sql, "heart_rate_coverage").iloc[0]
cohort_patients = int(cov["cohort_patients"])

print(f"cohort patients          : {cohort_patients:>9,}")
print(f"  ... with >=1 HR day    : {int(cov['patients_with_any_hr']):>9,}"
      f"   ({int(cov['patients_with_any_hr']) / cohort_patients:6.1%} of cohort)")
print(f"heart-rate days          : {int(cov['hr_days']):>9,}")

curve_sql = CTES + """
    SELECT rel_day, COUNT(DISTINCT person_id) AS n_patients
    FROM hr_daily
    WHERE rel_day BETWEEN -180 AND 365
    GROUP BY rel_day
    ORDER BY rel_day
"""

hr_curve = cached_gbq(curve_sql, "heart_rate_coverage_curve")

srp_common.plot_rel_day_curve(
    cfg, hr_curve, "n_patients",
    title="Patients with a Fitbit heart-rate record, by day relative to surgery",
    ylabel="patients with a heart-rate record", suppress=False,
    save_as="heart_rate_coverage")
srp_common.plot_rel_day_curve(
    cfg, hr_curve, "n_patients",
    title="Patients with a Fitbit heart-rate record, by day relative to surgery",
    ylabel="patients with a heart-rate record", suppress=True,
    save_as="heart_rate_coverage")

### How many patients have both a usable baseline and a usable follow-up?

As with steps, the paired count is the binding constraint on any return-to-baseline construction.
A day is counted as covered when the summary records any minutes at all, which is generous and
therefore an upper bound.

In [ ]:
window_case = srp_common.window_case_sql("rel_day")

paired_sql = CTES + f"""
    , per_window AS (
        SELECT
            person_id,
            {window_case} AS window_name,
            COUNT(DISTINCT rel_day) AS n_days
        FROM hr_daily
        WHERE {window_case} IS NOT NULL
        GROUP BY person_id, window_name
    ),
    wide AS (
        SELECT
            person_id,
            MAX(IF(window_name = 'baseline', n_days, 0)) AS baseline_days,
            MAX(IF(window_name = 'recovery', n_days, 0)) AS recovery_days
        FROM per_window
        GROUP BY person_id
    )
    SELECT
        COUNT(*)                                            AS patients_any_window,
        COUNTIF(baseline_days >= 7)                         AS baseline_7d,
        COUNTIF(recovery_days >= 7)                         AS recovery_7d,
        COUNTIF(baseline_days >= 7 AND recovery_days >= 7)  AS paired_7d
    FROM wide
"""

paired = cached_gbq(paired_sql, "heart_rate_paired_windows").iloc[0]
paired_7d = int(paired["paired_7d"])

for label in ["patients_any_window", "baseline_7d", "recovery_7d", "paired_7d"]:
    n = int(paired[label])
    flag = "  << below threshold, do not report" if 0 < n < SUPPRESS_THRESHOLD else ""
    print(f"{label:22s} {n:>9,}   ({n / cohort_patients:6.1%} of cohort){flag}")

# 3-signal

### What would deriving a resting heart rate actually cost, and does it move after surgery?

The resting value is taken as each patient-day's 5th percentile of minute-level readings, which
is the usual proxy when a device's own resting figure is not published. The query runs only if a
dry run prices it under the budget set in setup, so an accidental full scan cannot happen here.

In [ ]:
rhr_sql = CTES + f"""
    , minute_hr AS (
        SELECT
            m.person_id,
            DATE_DIFF(DATE(m.datetime), i.index_date, DAY) AS rel_day,
            m.heart_rate_value
        FROM `{CDR}`.heart_rate_minute_level AS m
        JOIN index_surgery AS i ON i.person_id = m.person_id
        WHERE DATE(m.datetime) BETWEEN DATE_SUB(i.index_date, INTERVAL 60 DAY)
                                   AND DATE_ADD(i.index_date, INTERVAL 180 DAY)
    ),
    daily_rhr AS (
        SELECT
            person_id,
            rel_day,
            COUNT(*) AS minutes,
            APPROX_QUANTILES(heart_rate_value, 100)[OFFSET(5)] AS resting_hr
        FROM minute_hr
        GROUP BY person_id, rel_day
        HAVING COUNT(*) >= 180
    )
    SELECT
        rel_day,
        COUNT(DISTINCT person_id)                          AS n_patients,
        APPROX_QUANTILES(resting_hr, 100)[OFFSET(25)]      AS p25_rhr,
        APPROX_QUANTILES(resting_hr, 100)[OFFSET(50)]      AS p50_rhr,
        APPROX_QUANTILES(resting_hr, 100)[OFFSET(75)]      AS p75_rhr
    FROM daily_rhr
    GROUP BY rel_day
    ORDER BY rel_day
"""

estimated_bytes = cfg.dry_run_bytes(rhr_sql)
rhr_curve = None

if 0 <= estimated_bytes <= MINUTE_LEVEL_BUDGET_GIB * 2**30:
    rhr_curve = cached_gbq(rhr_sql, "resting_heart_rate_curve")
    srp_common.plot_rel_day_curve(
        cfg, rhr_curve, "p50_rhr", band=("p25_rhr", "p75_rhr"),
        title="Median derived resting heart rate (IQR band), by day relative to surgery",
        ylabel="resting heart rate (bpm)", suppress=False, save_as="resting_heart_rate")
    srp_common.plot_rel_day_curve(
        cfg, rhr_curve, "p50_rhr", band=("p25_rhr", "p75_rhr"),
        title="Median derived resting heart rate (IQR band), by day relative to surgery",
        ylabel="resting heart rate (bpm)", suppress=True, save_as="resting_heart_rate")
elif estimated_bytes < 0:
    print("Dry run unavailable, so the scan is unpriced and the query was NOT run. "
          "Price it by hand before raising MINUTE_LEVEL_BUDGET_GIB or running it directly.")
else:
    print(f"Estimated scan {estimated_bytes / 2**30:,.1f} GiB exceeds the "
          f"{MINUTE_LEVEL_BUDGET_GIB} GiB budget -- narrow the day range or raise the budget "
          "deliberately. The cost itself is an argument against this metric.")

# 4-verdict

### What is the verdict for this metric?

The audit criteria are recorded as one row in the shared task cache, which is what
`metric_comparison.ipynb` reads to build the cross-metric scorecard. Nothing here is a
participant-level value: the row is counts, fractions and hand-entered judgements.

In [ ]:
srp_common.write_scorecard(
    cfg,
    family="wearable",
    source_table="heart_rate_summary + heart_rate_minute_level",
    index_rule=INDEX_RULE,
    outcome_type="continuous",
    cohort_patients=cohort_patients,
    patients_with_metric=int(cov["patients_with_any_hr"]),
    patients_analysable=paired_7d,
    coverage_share=paired_7d / cohort_patients,
    horizon_days=int(WINDOWS["recovery"][1]),
    derivation_priced_gib=(estimated_bytes / 2**30 if estimated_bytes >= 0 else None),
    informative_missingness="yes",
    leakage_risk="low",
    notes=("Resting value is derived, not published: the 5th percentile of minute-level "
           "readings on days with at least 180 minutes recorded. Coverage above is from the "
           "daily summary and is therefore an upper bound on the derived metric."),
)